# AIHub KorLectureSpeech (KlecSpeech) — EDA

`/data/ASR/RAW/AIHub_KorLectureSpeech/009.한국어_강의_데이터/01.데이터`

## 구조 (확인됨) — CounselingSpeech(KtelSpeech)와 동일한 ①형
- `{1.Training,2.Validation}/{라벨링데이터,원천데이터}_0908_add/KlecSpeech_<split>_D##_<label|wav>_#/D##/<G##|M##>/S######/`
- 세션 JSON: `dataSet.typeInfo{category,subcategory,place,inputType,speakers[1]}` + `dialogs[].{speaker,audioPath,textPath}`
- 발화별 txt(UTF-8 전사) + wav — **발화 단위 1:1 (2,418,428개)**
- **오디오 이미 16kHz mono PCM_16 → 복사만** (리샘플 불필요)
- 전사 KsponSpeech식: `자/`, `(3번)/(삼 번)` 확인 → 기존 정규화 재사용
- 규모: train 7,942세션 / **valid 94세션·32,971발화(~63h 추정)**
- 강의 특성: 세션당 화자 1명, age 지저분("30(?)", None)

In [3]:
from pathlib import Path
import re, json, random, numpy as np, pandas as pd
from collections import Counter
from IPython.display import Audio, display
try:
    import soundfile as sf
except ImportError:
    sf = None
    print("⚠ soundfile 없음")

DATA = Path("/data/ASR/RAW/AIHub_KorLectureSpeech/009.한국어_강의_데이터/01.데이터")

def label_to_audio(p):
    return Path(str(p).replace("라벨링데이터", "원천데이터").replace("_label_", "_wav_"))

def read_txt(p):
    return Path(p).read_text(encoding="utf-8", errors="replace").strip()

print("DATA 존재:", DATA.is_dir())

DATA 존재: True


## 1. 매니페스트 파서 (valid 전수 94세션 + train 표본)

> CounselingSpeech 파서 재사용. 전사는 발화별 txt에서.

In [4]:
TRAIN_SAMPLE_SESS = 50   # train 표본 세션 수

def build_manifest(label_root, split, max_sessions=None):
    rows = []
    jsons = sorted(label_root.rglob("S*.json"))
    if max_sessions:
        jsons = random.Random(0).sample(jsons, min(max_sessions, len(jsons)))
    for jp in jsons:
        try:
            ds = json.loads(jp.read_text(encoding="utf-8", errors="replace"))["dataSet"]
        except Exception:
            continue
        ti  = ds.get("typeInfo", {})
        spk = {s["id"]: s for s in ti.get("speakers", [])}
        audio_dir = label_to_audio(jp.parent)
        for d in ds.get("dialogs", []):
            sid = d.get("speaker"); sm = spk.get(sid, {})
            txt_name = Path(d["textPath"]).name
            wav_name = Path(d["audioPath"]).name
            tf = jp.parent / txt_name
            rows.append({
                "split": split, "session": jp.stem,
                "category": ti.get("category"), "subcat": ti.get("subcategory"),
                "place": ti.get("place"), "input": ti.get("inputType"),
                "utt": Path(txt_name).stem, "speaker": sid,
                "gender": sm.get("gender"), "age_raw": sm.get("age"),
                "text": read_txt(tf) if tf.exists() else "",
                "audio_path": str(audio_dir / wav_name),
            })
    df = pd.DataFrame(rows)
    if len(df):
        df["text"] = df["text"].astype("object")
    return df

df_v = build_manifest(DATA/"2.Validation/라벨링데이터_0908_add", "valid")              # 전수
df_t = build_manifest(DATA/"1.Training/라벨링데이터_0908_add", "train", TRAIN_SAMPLE_SESS)
print(f"valid 전수: 세션 {df_v['session'].nunique():,} / 발화 {len(df_v):,}")
print(f"train 표본: 세션 {df_t['session'].nunique():,} / 발화 {len(df_t):,}")
df_v.head(3)

valid 전수: 세션 81 / 발화 32,971
train 표본: 세션 48 / 발화 16,157


,split,session,category,subcat,place,input,utt,speaker,gender,age_raw,text,audio_path
0,valid,S000028,국어,초등1,studio,broadcast,000000,1,여,None,여러분 안녕하세요.,/data/ASR/RAW/AIHub_KorLectureSpeech/009.한국어_강...
1,valid,S000028,국어,초등1,studio,broadcast,000001,1,여,None,친구들이 독해의 열매를 맺어 독해력이 쑥+쑥 자랄 수 있도록 도와주는 혜은 선생님 ...,/data/ASR/RAW/AIHub_KorLectureSpeech/009.한국어_강...
2,valid,S000028,국어,초등1,studio,broadcast,000002,1,여,None,오늘의 생각 씨앗은요 (ㅎ)/(히읗) (ㅂ)/(비읍) 이에요. 선생님은 어렸을 때 ...,/data/ASR/RAW/AIHub_KorLectureSpeech/009.한국어_강...


In [5]:
# 매칭/결측 점검 (valid)
chk = df_v.sample(min(300, len(df_v)), random_state=0)
miss = int(chk["audio_path"].map(lambda p: not Path(p).exists()).sum())
print(f"오디오 존재 표본 {len(chk)} 중 없음: {miss}")
print(f"빈 전사: {int((df_v['text'].str.strip()=='').sum())}")
print(f"gender 결측: {int(df_v['gender'].isna().sum())}")
print("age_raw 값 분포:", df_v["age_raw"].value_counts(dropna=False).head(10).to_dict())

오디오 존재 표본 300 중 없음: 0
빈 전사: 0
gender 결측: 0
age_raw 값 분포: {'None': 10521, '30(?)': 6945, '30': 3438, '20(?)': 2506, '50(?)': 2316, '40': 1389, '60(?)': 1284, '40(?)': 1172, '60대': 674, '20': 638}


## 2. 분포 (valid 전수)

In [6]:
d = df_v
d["text_len"] = d["text"].str.len()
for col in ["category", "subcat", "gender", "place", "input"]:
    vc = d[col].value_counts(dropna=False)
    print(f"=== {col} ({d[col].nunique()}종) ===")
    print(vc.head(12).to_string()); print()
print("=== 전사 글자 수 ==="); print(d["text_len"].describe().round(1).to_string())
print(f"\n세션당 발화 수: 평균 {len(d)/d['session'].nunique():.0f}")

=== category (20종) ===
category
수학          6669
한국사         5919
국어          4982
과학(교과과목)    4307
사회(교과과목)    3632
사회(일반)      2005
교양           674
교육           638
과학(일반)       628
IT(일반)       567
예술           462
문학           367

=== subcat (14종) ===
subcat
고등1           6767
일반            6209
고등3           4682
일반(직업/자격증)    2963
고등2           2936
중학3           1877
중학2           1346
초등6           1308
중학1           1268
초등5           1061
초등3            825
초등4            767

=== gender (2종) ===
gender
남    19850
여    13121

=== place (1종) ===
place
studio    32971

=== input (1종) ===
input
broadcast    32971

=== 전사 글자 수 ===
count    32971.0
mean        52.7
std         37.8
min          3.0
25%         23.0
50%         42.0
75%         75.0
max        271.0

세션당 발화 수: 평균 407


## 3. 전사 컨벤션 (valid 전수) — KsponSpeech 규칙 재사용 가능한지 + 신규 패턴

In [7]:
txt = df_v["text"].fillna("")
print("특수문자 전수 (상위 25):")
print(txt.str.findall(r"[^가-힣a-zA-Z0-9\s]").explode().value_counts().head(25).to_string())
print(f"\n영문 포함: {txt.str.contains(r'[A-Za-z]').mean()*100:.2f}%  /  숫자 포함: {txt.str.contains(r'[0-9]').mean()*100:.2f}%")
print("\n기존 패턴 점검:")
for tag in ["b/", "n/", "l/", "o/", "u/", ")/(", ")(", "@", "(())"]:
    c = int(txt.str.contains(re.escape(tag)).sum())
    print(f"  '{tag}': {c:,}건")
# 단어 끝 부착 태그(LowQuality식)도 점검
suf = int(txt.str.contains(r"[가-힣][bnlou](?:$|[\s.,?!])").sum())
print(f"  단어끝 부착태그(했습니다n 등): {suf:,}건")
print("\n전사 샘플 8개:")
for t in txt.sample(min(8, len(txt)), random_state=1):
    print("  •", t[:85])

특수문자 전수 (상위 25):
text
(    54414
)    54411
/    48331
.    38091
,     8626
?     7003
+     1455
*      375
ㄱ      219
ㄴ      215
%      143
ㄷ      125
!      123
@       85
ㄹ       39
'       26
ㅁ       14
&        8
-        8
ㅣ        5
ㅇ        4
]        4
ㅂ        3
=        3
[        3

영문 포함: 29.22%  /  숫자 포함: 22.88%

기존 패턴 점검:
  'b/': 1,094건
  'n/': 4,985건
  'l/': 200건
  'o/': 515건
  'u/': 337건
  ')/(': 12,445건
  ')(': 242건
  '@': 53건
  '(())': 0건
  단어끝 부착태그(했습니다n 등): 0건

전사 샘플 8개:
  • 문화적 동질성과 일본에 보낸 국서 얘기를 해야겠죠.
  • 원주율을 나눗셈 식으로 쓰면 이렇게 쓸 수 (있어요)/(있어여). 원주 나누기 지름!
  • 않다는 뜻이 되잖아요 그죠, 이렇게 할때 기본적인 룰이라든지 규칙이라든지 정의에 대한 관념이 젓도록 적어도 서로 공유한다는 뜻입니다. 그 바탕,
  • 이/ 방군수포제가 시간이 지나면서 어떤 법으로 이제 나오게 되냐면
  • 그런데 법대 의대 너무 지나친 편중이 되고 있다 이런 말씀입니다. 이렇게 돼 갖고는 안 된다고 생각해요 이것이 거대한 낭비입니다 그렇지 않습니까?
  • 단면을 그을 때 (어떻게)/(어뜨케) 하니. 자/ 지형이 n/ 등장해야 돼요. 여기가 (300)/(삼백)과 여기 (200)/(이백)이니깐요. n/
  • n/ 이제 그거를 요런 (조그마한)/(쪼끄만한) 메시지 카드에다 써주시는 거야.
  • 수학선생인데,


## 3-1. 청취 — 강의 음향(스튜디오/방송) 확인

In [8]:
for r in df_v.sample(3, random_state=1).itertuples():
    print(f"[{r.category}/{r.gender}] {r.text[:70]}")
    if sf and Path(r.audio_path).exists():
        data, sr = sf.read(r.audio_path)
        display(Audio(data, rate=sr))
    else:
        print("   (오디오 없음)")

[한국사/남] 문화적 동질성과 일본에 보낸 국서 얘기를 해야겠죠.


[수학/남] 원주율을 나눗셈 식으로 쓰면 이렇게 쓸 수 (있어요)/(있어여). 원주 나누기 지름!


[철학/남] 않다는 뜻이 되잖아요 그죠, 이렇게 할때 기본적인 룰이라든지 규칙이라든지 정의에 대한 관념이 젓도록 적어도 서로 공유한다는 뜻


## 4. 오디오 속성 (valid 표본) + duration 측정 부하 확인

In [9]:
samp = df_v.sample(min(100, len(df_v)), random_state=0)
rows = []
for r in samp.itertuples():
    try:
        i = sf.info(r.audio_path)
        rows.append((i.samplerate, i.channels, i.subtype, round(i.frames/i.samplerate, 3)))
    except Exception as e:
        rows.append(("ERR", str(e)[:20], "", None))
a = pd.DataFrame(rows, columns=["sr","ch","subtype","dur"])
print("sample_rate :", a["sr"].value_counts().to_dict())
print("channels    :", a["ch"].value_counts().to_dict())
print("subtype     :", a["subtype"].value_counts().to_dict())
d = a["dur"].dropna()
print(f"길이(초): 평균 {d.mean():.2f} / 중앙 {d.median():.2f} / p95 {d.quantile(0.95):.2f} / 최대 {d.max():.2f}")
print(f"valid 총시간 추정: {d.mean()*len(df_v)/3600:.1f}h")
print("\n※ JSON에 duration 없음 → 빌드 때 sf.info(헤더만)로 측정 (33k개, 수 분이면 충분)")

sample_rate : {16000: 100}
channels    : {1: 100}
subtype     : {'PCM_16': 100}
길이(초): 평균 6.73 / 중앙 5.97 / p95 13.82 / 최대 24.48
valid 총시간 추정: 61.7h

※ JSON에 duration 없음 → 빌드 때 sf.info(헤더만)로 측정 (33k개, 수 분이면 충분)


## 5. 화자/세션 — 강의는 세션=화자

> speakers id가 세션 내 '1'뿐 → 화자 추적은 세션 단위. train/valid 강사 겹침은 메타로 확인 불가(이름 없음).

In [10]:
print(f"세션 수: {df_v['session'].nunique()} / 세션당 화자 수 분포: {df_v.groupby('session')['speaker'].nunique().value_counts().to_dict()}")
print(f"카테고리별 세션 수:")
print(df_v.groupby("category")["session"].nunique().to_string())
print("\n성별 세션 수:")
print(df_v.groupby("gender")["session"].nunique().to_string())

세션 수: 81 / 세션당 화자 수 분포: {1: 66, 2: 12, 4: 2, 3: 1}
카테고리별 세션 수:
category
IT(일반)         1
IT(직업/자격증)     1
경영             1
과학(교과과목)      13
과학(일반)         1
교양             1
교육             1
국어            17
금융             1
기술             1
기타             1
문학             1
사회(교과과목)      12
사회(일반)         3
수학            18
예술             1
인문             1
전문자격           1
철학             1
한국사           13

성별 세션 수:
gender
남    49
여    44


In [11]:
# ============================================================
# 비식별화(PII) 토큰 포함 발화 → 전사 + 음성 직접 청취  (모든 EDA 노트북 공용)
# 파서로 DataFrame을 만든 셀을 먼저 실행한 뒤, 이 셀을 새 셀에 붙여 실행.
# DataFrame 변수(df/df_v/...)와 오디오 경로 컬럼(audio_path/wav/...)을 자동 탐지.
# ============================================================
import re
from pathlib import Path
from IPython.display import Audio, display
try:
    import soundfile as sf
except ImportError:
    sf = None
    print("⚠ soundfile 없음 → conda activate TRAIN-ASR")

import pandas as pd

# ---------- 설정 ----------
N_LISTEN = 5          # 들어볼 발화 수
RANDOM_STATE = 0      # None이면 매번 다른 표본
# 비식별화(PII) 토큰: 음향 토큰(b/ n/ l/ o/ u/)과 구분되는 익명화 전용 패턴
PII_PATTERNS = {
    "@ (이름 마커)":           re.compile(r"@"),
    "ㅇㅇ류 (익명화 2자+)":     re.compile(r"ㅇ{2,}"),
    "name/ (이름 태그)":        re.compile(r"(?:^|\s)name/", re.I),
    "[마스킹]":                re.compile(r"\[[^\]]{0,15}\]"),
    "<마스킹>":                re.compile(r"<[^>]{0,15}>"),
    "*** (별표 2+)":           re.compile(r"\*{2,}"),
    "xxx (엑스 2+)":           re.compile(r"[xX]{2,}"),
    "○○ (공백원 2+)":          re.compile(r"[○◯]{2,}"),
}

# ---------- 1) text DataFrame 자동 탐지 ----------
def _find_text_frame():
    g = globals()
    for name in ["df_v","df_valid","df","df_t","df_train","d","a"]:
        o = g.get(name)
        if isinstance(o, pd.DataFrame) and "text" in o.columns:
            return name, o
    for name, o in g.items():
        if not name.startswith("_") and isinstance(o, pd.DataFrame) and "text" in o.columns:
            return name, o
    return None, None

_name, _df = _find_text_frame()
if _df is None:
    raise RuntimeError("text 컬럼 DataFrame 없음 — 파서 셀을 먼저 실행하세요.")

# ---------- 2) 오디오 경로 컬럼 자동 탐지 ----------
AUDIO_COL = next((c for c in ["audio_path","wav","src_wav","audio","path","filepath"]
                  if c in _df.columns), None)
print(f"[대상] DataFrame '{_name}' · {len(_df):,} 발화 · 오디오 컬럼: {AUDIO_COL or '없음(PCM 직접 노트북일 수 있음)'}\n")

# ---------- 3) PII 토큰 집계 ----------
_txt = _df["text"].fillna("").astype("object")
print("=== 비식별화 토큰 집계 (text 원본) ===")
present = []
for label, pat in PII_PATTERNS.items():
    occ = int(_txt.str.count(pat).sum())
    utt = int(_txt.str.contains(pat).sum())
    if occ:
        present.append((label, pat, utt, occ))
        print(f"  {label:20s} 발화 {utt:>6,} · 출현 {occ:>6,} ({utt/len(_df)*100:.3f}%)")
if not present:
    print("  → 이 데이터셋엔 정의된 비식별화 토큰이 없음 (자유발화/낭독 등). 청취 생략.")

# ---------- 4) PII 포함 발화만 필터 → 전사 + 음성 재생 ----------
if present and AUDIO_COL:
    mask = pd.Series(False, index=_df.index)
    for _, pat, _, _ in present:
        mask |= _txt.str.contains(pat)
    hits = _df[mask]
    print(f"\n=== 비식별화 토큰 포함 발화 {len(hits):,}건 중 {min(N_LISTEN,len(hits))}개 청취 ===")
    print("   (전사의 마스킹 부분이 음성에서 실제로 어떻게 발화되는지 직접 확인)\n")
    sample = hits.sample(min(N_LISTEN, len(hits)), random_state=RANDOM_STATE)
    for r in sample.itertuples():
        text = getattr(r, "text", "")
        ap = getattr(r, AUDIO_COL, None)
        # 어떤 PII 패턴에 걸렸는지 표시
        tags = [lab for lab, pat, _, _ in present if pat.search(text or "")]
        print(f"[{', '.join(tags)}]")
        print(f"  전사: {text[:100]}")
        if sf and ap and Path(str(ap)).exists():
            try:
                data, sr = sf.read(str(ap))
                display(Audio(data, rate=sr))
            except Exception as e:
                print(f"   (재생 실패: {str(e)[:50]})")
        else:
            print(f"   (오디오 경로 없음/미존재: {ap})")
        print()
elif present and not AUDIO_COL:
    print("\n⚠ PII 토큰은 있으나 오디오 경로 컬럼을 못 찾음.")
    print("  이 노트북이 PCM을 직접 읽는 방식이면, 아래처럼 수동 지정:")
    print("  → sample = _df[mask].sample(N_LISTEN); 각 행의 키로 원본 PCM 경로를 구성해 재생")

[대상] DataFrame 'df_v' · 32,971 발화 · 오디오 컬럼: audio_path

=== 비식별화 토큰 집계 (text 원본) ===
  @ (이름 마커)            발화     53 · 출현     85 (0.161%)
  [마스킹]                발화      3 · 출현      3 (0.009%)
  xxx (엑스 2+)          발화      1 · 출현      1 (0.003%)

=== 비식별화 토큰 포함 발화 57건 중 5개 청취 ===
   (전사의 마스킹 부분이 음성에서 실제로 어떻게 발화되는지 직접 확인)

[@ (이름 마커)]
  전사: n/ 안녕하세요. 반갑습니다. 여러분의 국사 완성자 @/류성완 입니다. 아이. 너무+ 너무 반갑습니다. 아이. 여러분 좋아요. 드디어 우리 조선 경제 들어오게 되었습니다.



[[마스킹]]
  전사: 이거 직각 (3각형)/(삼각형)이네요. 그리고 (3점)/(세 점) (D)/(디) (E)/(이) (F)/(에프) 는 접점이라고 하고 선분 (A)/(에이) (B)/(비) (A)/(에이



[@ (이름 마커)]
  전사: @/ 소정묘 의 경우는 거악입니다. @/ 소정묘 를 내버려두고 나서,



[@ (이름 마커)]
  전사: @/ 용찰 @/ 철수 야/ @/ 철수야.



[@ (이름 마커)]
  전사: n/ @/ 석낭중 일단 쓸까요?
